In [ ]:
from pybaseball import statcast, cache
import pandas as pd

from src.data import pull_statcast
import config

In [ ]:
statcast_df = pull_statcast(config.SEASON)
statcast_df.head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
4440,FF,2025-09-28,91.3,1.93,6.21,"Aldegheri, Sam",805904,691951,None,ball,...,<NA>,1.31,0.35,0.35,41.3,<NA>,<NA>,<NA>,<NA>,<NA>
4188,FF,2025-09-28,93.1,2.08,6.12,"Aldegheri, Sam",805904,691951,None,called_strike,...,<NA>,1.19,0.72,0.72,37.8,<NA>,<NA>,<NA>,<NA>,<NA>
4135,FF,2025-09-28,91.8,2.06,6.26,"Aldegheri, Sam",805904,691951,None,ball,...,<NA>,1.3,0.41,0.41,37.8,<NA>,<NA>,<NA>,<NA>,<NA>
3894,FF,2025-09-28,92.5,2.05,6.2,"Aldegheri, Sam",805904,691951,None,ball,...,<NA>,1.17,0.36,0.36,39.2,<NA>,<NA>,<NA>,<NA>,<NA>
3823,FF,2025-09-28,92.7,1.99,6.21,"Aldegheri, Sam",805904,691951,None,called_strike,...,<NA>,1.22,0.54,0.54,39.7,<NA>,<NA>,<NA>,<NA>,<NA>


In [4]:
filtered_df = statcast_df[['pitch_type', 
                           'release_speed', 
                           'pfx_x', 'pfx_z', # horz / vert movement
                           'plate_x', 'plate_z',
                           'balls', 'strikes',
                           'inning', 'outs_when_up', 
                           'on_1b', 'on_2b', 'on_3b', # runners on
                           'stand', # 'L' or 'R'
                           'pitch_number'
                           ]]
filtered_df.shape

(756325, 15)

In [5]:
filtered_df['stand'].value_counts()

stand
R    419348
L    336977
Name: count, dtype: int64

In [6]:
from pybaseball import statcast_pitcher_pitch_arsenal
# percentage breakdown of pitch types
arsenal_df = statcast_pitcher_pitch_arsenal(2026, arsenal_type="n_")
arsenal_df.head()

,"last_name, first_name",pitcher,n_ff,n_si,n_fc,n_sl,n_ch,n_cu,n_fs,n_kn,n_st,n_sv
0,"Alcantara, Sandy",645261,18.9,24.9,13.9,7.8,22.4,4.1,NaN,NaN,7.9,NaN
1,"Sánchez, Cristopher",650911,NaN,42.7,NaN,18.3,39.0,NaN,NaN,NaN,NaN,NaN
2,"Soriano, José",667755,23.4,26.2,NaN,5.3,NaN,25.2,19.9,NaN,NaN,NaN
3,"Gausman, Kevin",592332,52.0,NaN,NaN,9.4,NaN,NaN,38.6,NaN,NaN,NaN
4,"Luzardo, Jesús",666200,25.2,17.5,NaN,NaN,21.0,NaN,NaN,NaN,36.2,NaN


In [7]:
arsenal_df.columns


Index(['last_name, first_name', 'pitcher', 'n_ff', 'n_si', 'n_fc', 'n_sl',
       'n_ch', 'n_cu', 'n_fs', 'n_kn', 'n_st', 'n_sv'],
      dtype='object')

In [8]:
statcast_df["pitch_type"].value_counts()

pitch_type
FF    237316
SI    115375
SL    107554
CH     76338
ST     56510
FC     55902
CU     49854
FS     24674
KC     13251
SV      3738
EP       987
FA       901
FO       828
CS       409
KN       140
PO        55
UN        14
SC         7
Name: count, dtype: int64

statcast_pitcher_pitch_arsenal only returns 10 kinds of pitches. Notably not knuckle curve, forkball, and some others.

In [9]:
normal_pitch_types = list(arsenal_df.drop(["last_name, first_name", "pitcher"], axis=1).columns)
normal_pitch_types = list(map(lambda x: x.replace("n_", "").upper(), normal_pitch_types))
other_pitches = statcast_df[~statcast_df["pitch_type"].isin(normal_pitch_types)]
other_pitches["pitch_type"].value_counts()
'''
pitch_type
KC    13251 - knuckle curve (collapse with regular curve?)
EP      987 - eephus (position players pitching)
FA      901 - other (more position players?)
FO      828 - forkball (roki sasaki & kodai senga)
CS      409 - slow curve (collapse with regular curve or drop entirely)
PO       55 - pitchout (drop)
UN       14 - unknown (drop)
SC        7 - screwball (collapse with changeup?)
Name: count, dtype: int64
'''

'\npitch_type\nKC    13251 - knuckle curve (collapse with regular curve?)\nEP      987 - eephus (position players pitching)\nFA      901 - other (more position players?)\nFO      828 - forkball (roki sasaki & kodai senga)\nCS      409 - slow curve (collapse with regular curve or drop entirely)\nPO       55 - pitchout (drop)\nUN       14 - unknown (drop)\nSC        7 - screwball (collapse with changeup?)\nName: count, dtype: int64\n'

In [10]:
pitch_profiles = (
    statcast_df.groupby(["pitcher", "pitch_type"])
    [["release_speed", "pfx_x", "pfx_z", "release_spin_rate"]]
    .mean()
)

pitch_profiles

release_speed     pfx_x     pfx_z  release_spin_rate
pitcher pitch_type                                                      
434378  CH              84.678829     -1.08  0.780766        1728.887387
        CU              78.470079  0.628504 -1.027428        2748.244094
        FF               93.92132  -0.73945  1.552868        2434.958545
        SI                  93.27    -0.993     1.362             2474.0
        SL               87.12883  0.337743  0.459934        2503.444811
...                           ...       ...       ...                ...
829272  FF              91.089272  0.582874  1.360575        2202.819923
        KC              72.334109 -0.594109 -1.654961        2731.852713
        SI              90.366667  1.273333  1.061667        2171.666667
        SL                81.6248     -0.21   0.25624            2381.12
        ST              76.533333 -1.053333 -0.886667        2746.666667

[4569 rows x 4 columns]

In [11]:
whiff_descriptions = {
    "swinging_strike", "swinging_strike_blocked", "missed_bunt"
}
foul_descriptions = {
    "foul", "foul_tip", "foul_bunt"
}

statcast_df["whiff"] = statcast_df["description"].isin(whiff_descriptions)
statcast_df["foul"] = statcast_df["description"].isin(foul_descriptions)

statcast_df["whiff"].value_counts()

whiff
False    672745
True      83580
Name: count, dtype: int64

In [12]:
statcast_df["foul"].value_counts()

foul
False    612395
True     143930
Name: count, dtype: int64

In [13]:
statcast_df["pitch_type"].value_counts()

pitch_type
FF    237316
SI    115375
SL    107554
CH     76338
ST     56510
FC     55902
CU     49854
FS     24674
KC     13251
SV      3738
EP       987
FA       901
FO       828
CS       409
KN       140
PO        55
UN        14
SC         7
Name: count, dtype: int64